# 03. Train — U-Net + ResNet-50 학습

## Google Colab 환경 설정

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules or "/kaggle" in sys.executable or __import__("os").path.exists("/kaggle/input")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations optuna

elif IN_KAGGLE:
    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations optuna


## Imports

In [ ]:
import os
import sys

sys.path.append("..")

import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.model import build_model


## Config

In [ ]:
if IN_COLAB:
    SPLITS_DIR = "/content/drive/MyDrive/data/splits"
    CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
    DATA_ROOT = "/content/drive/MyDrive/data/OCT5k"
elif IN_KAGGLE:
    SPLITS_DIR = "/kaggle/input/oct5k-checkpoints/splits"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    DATA_ROOT = "/kaggle/input/oct5k-data/OCT5k"
else:
    SPLITS_DIR = "../data/splits"
    CHECKPOINT_DIR = "../checkpoints"
    DATA_ROOT = "../data/OCT5k"

import os

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.join(CHECKPOINT_DIR, "search"), exist_ok=True)

N_FOLDS = 5
N_TRIALS = 20
SEARCH_EPOCHS = 10
FINAL_EPOCHS = 30


## 모델 구조 확인

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model()
n_params = sum(p.numel() for p in model.parameters())

print("device:", device)
print(f"파라미터 수: {n_params:,}")
del model


## Optuna 하이퍼파라미터 탐색 (고정 Split 기준)

In [ ]:
!python -m src.tune \
    --train-csv {SPLITS_DIR}/train.csv --val-csv {SPLITS_DIR}/val.csv \
    --n-trials {N_TRIALS} --search-epochs {SEARCH_EPOCHS} --final-epochs {FINAL_EPOCHS} \
    --search-checkpoint-dir {CHECKPOINT_DIR}/search --final-checkpoint-dir {CHECKPOINT_DIR} \
    --run-name fixed


## K-Fold 학습 (Optuna가 찾은 최적 파라미터 재사용)

In [ ]:
trials_df = pd.read_csv(os.path.join(CHECKPOINT_DIR, "search", "optuna_trials.csv"))
best_row = trials_df.loc[trials_df["value"].idxmax()]

best_lr = best_row["params_lr"]
best_batch_size = int(best_row["params_batch_size"])
best_weight_decay = best_row["params_weight_decay"]

print(f"최적 lr={best_lr}, batch_size={best_batch_size}, weight_decay={best_weight_decay}")

for fold_idx in range(N_FOLDS):
    !python -m src.train \
        --train-csv {SPLITS_DIR}/fold{fold_idx}_train.csv --val-csv {SPLITS_DIR}/fold{fold_idx}_val.csv \
        --epochs {FINAL_EPOCHS} --batch-size {best_batch_size} --lr {best_lr} --weight-decay {best_weight_decay} \
        --checkpoint-dir {CHECKPOINT_DIR} --run-name fold{fold_idx}


## 결과 비교 표 (고정 Split vs K-Fold)

In [ ]:
def best_metrics(history_path):
    h = pd.read_csv(history_path)
    best = h.loc[h["val_dice"].idxmax()]
    return best["val_dice"], best["val_iou"]


fixed_dice, fixed_iou = best_metrics(os.path.join(CHECKPOINT_DIR, "fixed_history.csv"))

fold_dices, fold_ious = [], []
for fold_idx in range(N_FOLDS):
    dice, iou = best_metrics(os.path.join(CHECKPOINT_DIR, f"fold{fold_idx}_history.csv"))
    fold_dices.append(dice)
    fold_ious.append(iou)

import numpy as np

comparison = pd.DataFrame(
    [
        {"split": "고정 Split (1회)", "val_dice": fixed_dice, "val_dice_std": 0.0, "val_iou": fixed_iou, "val_iou_std": 0.0},
        {
            "split": f"K-Fold ({N_FOLDS}-fold 평균)",
            "val_dice": np.mean(fold_dices),
            "val_dice_std": np.std(fold_dices),
            "val_iou": np.mean(fold_ious),
            "val_iou_std": np.std(fold_ious),
        },
    ]
)
comparison


## 결과 비교 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.boxplot(fold_dices, positions=[1], widths=0.4, labels=[f"K-Fold ({N_FOLDS}개)"])
ax.scatter([1] * N_FOLDS, fold_dices, color="gray", alpha=0.6, zorder=3, label="fold별 val_dice")
ax.axhline(fixed_dice, color="red", linestyle="--", label=f"고정 Split = {fixed_dice:.3f}")

ax.set_ylabel("val_dice")
ax.set_title("고정 Split vs K-Fold val_dice 비교")
ax.legend()
plt.tight_layout()
plt.show()
